In [1]:
%run pi1.ipynb
%run monodromyfunctions.ipynb
%run ptolemy_equations.ipynb\
%run Matrix.ipynb

def apply_dict_to_str(dictionary:dict, word:str):
    for key in dictionary.keys():
        word = word.replace(key, dictionary[key])
    return word




#This class will compute all boundary parabolic representations of the given PTB and tensor them with the abelianization homomorphism.
class Representations():
    def __init__(self, monodromy:str):
        self.monodromy = monodromy
        self.pi1 = Pi1(monodromy)
        self.h1 = self.pi1.homology
        
        self.ptolemy = PtolemyEquations(self.monodromy)
        self.coord_ring = self.ptolemy.list_all_coord_ring() 
        
        
        self.ptolemy_coordinates_ring = LaurentPolynomialRing(QQ, 'c', len(self.monodromy)+1)
        self.h1_group_ring = self.h1.algebra(self.ptolemy_coordinates_ring)
        
        
        temp_gluing_dict = self.ptolemy.gluing_dict
        temp_gluing_dict['t_0 '] = '1'
        self.gluing_dict = temp_gluing_dict
        
        
        
        
        
        #for the sake of cleaning up some of this code, we refer to each obstruction class in an array as:
        #0 - trivial
        #1 - monodromy
        #2 - meridean
        #3 - longitude
        #4 - mon-mer
        #5 - mon-long
        #6 - mer-long
        #7 - all
    
    
    def make_matrix_space(self, ring):
        
        return MatrixSpace(ring, 2, 2)
    
    
    def repeat_gluing_dict(self, word:str):
        end = len(self.monodromy)
    
        for w in range(end):
            word = apply_dict_to_str(self.gluing_dict, word)
        
        return word
    
    #Make a dictionary which replaces string values with entries in the ring. This allows us to cast the matrix of strings into a matrix of ring elements.
    
    def string_to_ring_dict(self, ring):
        end = len(self.monodromy) + 1
        dictionary = {}
        
        
        for w in range(end):
            dictionary[f't_{w} '] = {ring.gens()[w]}
        return dictionary
    
    
    
    def replace_in_matrix(self, matrix):
        end = len(self.monodromy) - 1

        for i in range(2):
            for j in range(2):
                matrix[i][j] = self.repeat_gluing_dict(matrix[i][j])
            
        return matrix
    
    def make_generic_mer_representation(self):
        end = len(self.monodromy) - 1
        R = self.ptolemy_coordinates_ring
        
        mat = self.replace_in_matrix([[f'1 + (b_0 )', f'1'], [f'(t_{end} )', '0']])

        return matrix(self.h1_group_ring, [[R(mat[0][0])*R(self.repeat_gluing_dict('v_0 '))^-1, R(f'-t_{end}')^-1],[R(f't_{end}'), 0]]) * self.h1_group_ring.gen(0)
    
    
    def make_generic_long_representation(self):
        end = len(self.monodromy) - 1
        R = self.ring
        
        mat = self.replace_in_matrix([['1', '-(t_0 )*(b_0 ) - (h_0 )^2'], ['(v_0 )', '-(b_0 )']])
        
        return matrix(self.h1_group_ring, [[R(f't_{end}')^-1, R(mat[0][1])*R(f't_{end}')^(-2)],[R(mat[1][0]), R(mat[1][1])*R(f't_{end}')^-1]]) * self.h1_group_ring.gen(1)
    
    #For the monodromy representation, when we transfer from R to L or L to R, we pick up a matrix. This will be an upper right triangular matrix with 1s on the diagonal.
    
    def hinge(self, index):
        R = self.ptolemy_coordinates_ring
        sign = 0
        if self.monodromy[index] == 'R':
            sign = 1
        else:
            sign = -1
        
        return matrix(self.h1_group_ring, [[1, sign * R(self.repeat_gluing_dict(f'b_{index} '))*R(self.repeat_gluing_dict(f'h_{index} * v_{index} '))^-1],[0, 1]])
    
    def compare_to_last(self, index):
        return self.monodromy[index] != self.monodromy[index - 1]
        
    def make_generic_monodromy_representation(self):
        end = len(self.monodromy)
        
        monodromy = identity_matrix(2)
        
        for w in range(end):
            if self.compare_to_last(w):
                monodromy = self.hinge(w) * monodromy
        
        return monodromy * self.h1_group_ring.gen(2)
        

In [114]:
%run pi1.ipynb

#Function to merge two dictionaries
def merge(dict1, dict2):
    return dict(zip(list(dict1.keys()) + list(dict2.keys()), list(dict1.values()) + list(dict2.values())))

#Function to generate list of variables
def make_variables(monodromy):
    end = len(monodromy)
    variables = ['m','l','t']
    
    return variables + [f'c{ptol}' for ptol in range(end)]

#Function to initialize ring with appropriate number of elements   
def make_ring(monodromy):
    
    return LaurentPolynomialRing(QQ, make_variables(monodromy))
    
#Each layer in the triangulation gives a gluing dependent on the move.
def move_gluing(monodromy, step:int, R):
    if monodromy[step] == 'R':
        return merge({f'h_{step+1}' : f'h_{step}', f'b_{step+1}' : f'v_{step}', f'v_{step+1}' : R.gens()[step+3]} , {f'(-1)*h_{step+1}' : f'(-1)*h_{step}', f'(-1)*b_{step+1}' : f'(-1)*v_{step}', f'(-1)*v_{step+1}' : -R.gens()[step+3]})
    elif monodromy[step] == 'L':
        return merge({f'h_{step+1}' : R.gens()[step+3], f'b_{step+1}' : f'(-1)*h_{step}', f'v_{step+1}' : f'v_{step}'} , {f'(-1)*h_{step+1}' : -R.gens()[step+3], f'(-1)*b_{step+1}' : f'h_{step}', f'(-1)*v_{step+1}' : f'(-1)*v_{step}'})
    

#This dictionary effectively shifts all edges down one layer in the layered triangulation.    
def pre_gluing_dict(monodromy, R):
    end = len(monodromy) - 1
    if monodromy[-1] == 'L':
        gluing_dict = merge({'h_0' : R.gens()[-1], 'v_0' : f'v_{end}', 'b_0' : f'(-1)*h_{end}'}, {'(-1)*h_0' : -R.gens()[-1], '(-1)*v_0' : f'(-1)*v_{end}', '(-1)*b_0' : f'h_{end}'})
    else:
        gluing_dict = merge({'h_0' : f'h_{end}', 'v_0' : R.gens()[-1], 'b_0' : f'v_{end}'}, {'(-1)*h_0' : f'(-1)*h_{end}', '(-1)*v_0' : -R.gens()[-1], '(-1)*b_0' : f'(-1)*v_{end}'})
    for w in range(end):
        gluing_dict = merge(gluing_dict, move_gluing(monodromy,w, R))
        
    minus_gens = [-gen for gen in R.gens()]
    gluing_dict = merge(dict(zip(minus_gens, minus_gens)) , gluing_dict)
    gluing_dict = merge(dict(zip(R.gens(), R.gens())) , gluing_dict)
    return gluing_dict

#Continue shifting all edges down until all edges are identified with a top edge.
def gluing_dict(monodromy, R):
    end = len(monodromy)
    
    dic = pre_gluing_dict(monodromy, R)
    newvals = list(dic.values())
    for rep in range(end):
        newvals = [dic[val] for val in newvals]
    
    return dict(zip(list(dic.keys()), newvals))


#Generate a matrix associated to each of 'm', 'l', 't'
def make_generic_mer_representation(monodromy, R):
    dic = gluing_dict(monodromy, R)

    return matrix([[(1 + dic['b_0'])/dic['v_0'], -1/R.gens()[-1]], [R.gens()[-1], 0]]) * m

    
def make_generic_long_representation(monodromy, R):
    dic = gluing_dict(monodromy, R)

    return matrix([[1/R.gens()[-1], (-c0*dic['b_0'] - dic['h_0']^2)/R.gens()[-1]^2], [dic['v_0'], -dic['b_0']/R.gens()[-1]]]) * l


    
#For the monodromy representation, when we transfer from R to L or L to R, we pick up a matrix. This will be an upper right triangular matrix with 1s on the diagonal.
    
def hinge(monodromy, index, R):
    dic = gluing_dict(monodromy, R)
    sign = 0
    if monodromy[index] == 'R':
        sign = 1
    else:
        sign = -1

    return matrix([[1, sign * dic[f'b_{index}']* (dic[f'h_{index}'] * dic[f'v_{index}'])^-1],[0, 1]])

def compare_to_last(monodromy, index):
    return monodromy[index] != monodromy[index - 1]

def make_generic_monodromy_representation(monodromy, R):
    end = len(monodromy)
    rep = identity_matrix(2)

    for w in range(end):
        if compare_to_last(monodromy, w):
            rep = hinge(monodromy, w, R) * rep

    return rep * t

class Representation():
    
    def __init__(self,monodromy):
        S.<a,b,x> = LaurentPolynomialRing(QQ) 
        
        self.monodromy = monodromy
        self.R = make_ring(self.monodromy)
        self.R.inject_variables()
        self.M = MatrixSpace(R, 2)
        self.S = S
        self.gluing = gluing_dict(monodromy, self.R)
        
        self.meridean_rep = make_generic_mer_representation(self.monodromy, self.R)
        self.longitude_rep = make_generic_long_representation(self.monodromy, self.R)
        self.monodromy_rep = make_generic_monodromy_representation(self.monodromy, self.R)
        
        self.representation = self.S.hom([self.meridean_rep, self.longitude_rep, self.monodromy_rep], self.M)
    
    def __call__(self, word):
        
        return None
    
    

In [102]:
monodromy = 'RLLLRRRL'
R = make_ring(monodromy)
S.<a,b,x> = LaurentPolynomialRing(QQ)
R.inject_variables()
gluing_dict(monodromy, R)

f = S.hom([m, l, t], R)

f

Defining m, l, t, c0, c1, c2, c3, c4, c5, c6, c7


Ring morphism:
  From: Multivariate Laurent Polynomial Ring in a, b, x over Rational Field
  To:   Multivariate Laurent Polynomial Ring in m, l, t, c0, c1, c2, c3, c4, c5, c6, c7 over Rational Field
  Defn: a |--> m
        b |--> l
        x |--> t

In [115]:
Representation('RLRL')

Defining m, l, t, c0, c1, c2, c3


ValueError: relations do not all (canonically) map to 0 under map determined by images of generators

In [98]:
type(R)

<class 'sage.rings.polynomial.laurent_polynomial_ring.LaurentPolynomialRing_mpair_with_category'>

In [37]:
ring.gens()

(m, l, t, c0, c1, c2, c3, c4)